# derived_8.4-eval-mlp-1.0 — MLP vs XGBoost on the derived_8.4 Split

This experiment replaces the XGBoost experts of `derived_8.4-eval-1.1` with PyTorch **MLP** regressors to test whether a neural net can match or beat XGBoost — motivated by the hypothesis that XGBoost cannot extrapolate to feature values unseen during training. Two model families are evaluated on the Washington-only `derived_8.4` split (7 stations; trainval 2017–2022, test 2023–2025, 6,620 test samples):

- **1-regime**: a single global MLP on the 54-feature shared backbone (`shared_backbone_54`).
- **2-regime**: the best cluster config from eval-1.1 — `Clustering_V0_Full_k2` winner (c0=0, c1=10): cluster 0 uses the 54 backbone features, cluster 1 adds 10 delta features (64 total). Regime labels come from the same KMeans(k=2) router on the 50 V0 features (seed 42).

A 27-config hyperparameter sweep runs in **both** families (8 workers in parallel on an H100). All MLPs train on trainval with early stopping on a temporal 10% holdout (each station's last rows by date, mirroring the test split), and are evaluated on the untouched 2023–2025 test set. XGBoost reference rows are loaded from `derived_8.4-eval-1.1`.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Robust resolution of the experiment dir whether executed from notebooks/ or in-place.
candidates = [Path.cwd() / "experiment/derived_8.4-eval-mlp-1.0", Path.cwd()]
EXP_DIR = next((p for p in candidates if (p / "metrics_summary.csv").exists()), Path.cwd())

df_summary = pd.read_csv(EXP_DIR / "metrics_summary.csv")
df_per_regime = pd.read_csv(EXP_DIR / "per_regime_metrics_summary.csv")
df_sweep = pd.read_csv(EXP_DIR / "sweep_results.csv")
df_ood = pd.read_csv(EXP_DIR / "ood_summary.csv")
df_timing = pd.read_csv(EXP_DIR / "timing_summary.csv")
with open(EXP_DIR / "selected_features.json") as f:
    selected_meta = json.load(f)
with open(EXP_DIR / "timing_log.json") as f:
    timing_log = json.load(f)

print(f"Loaded {len(df_summary)} leaderboard rows, {len(df_sweep)} sweep rows, "
      f"{len(df_per_regime)} per-regime rows.")
print(f"MLP winners per family: {selected_meta['mlp_winners']}")

Loaded 12 leaderboard rows, 54 sweep rows, 12 per-regime rows.
MLP winners per family: {'1regime': 'w1024x1024', '2regime': 'w256x256_d0.3'}


## Overall Model Leaderboard

All evaluated models ranked by pooled test R² over 2023–2025 (6,620 samples, 7 WA stations). MLP rows carry the sweep `config_id`; XGBoost rows are the eval-1.1 references.

In [2]:
cols = ["model_name", "strategy_name", "pooled_r2", "pooled_rmse", "pooled_ubrmse", "pooled_bias", "pooled_mae", "pooled_pearson"]
print("### Overall Leaderboard (2023-2025 Test Set)")
print(df_summary[cols].to_markdown(index=False))

### Overall Leaderboard (2023-2025 Test Set)
| model_name                                 | strategy_name             |   pooled_r2 |   pooled_rmse |   pooled_ubrmse |   pooled_bias |   pooled_mae |   pooled_pearson |
|:-------------------------------------------|:--------------------------|------------:|--------------:|----------------:|--------------:|-------------:|-----------------:|
| Clustering_V0_Full_k2 (Winner c0=0, c1=10) | XGBoost_Reference         |    0.81496  |     0.0438196 |       0.043337  |    0.00648567 |    0.0337195 |         0.905594 |
| Global Single Model (54 Backbone)          | XGBoost_Reference         |    0.77923  |     0.0478636 |       0.0466868 |    0.0105484  |    0.0370592 |         0.889432 |
| MLP 2-Regime (w256x256_d0.3)               | MLP_Clustering_V0_Full_k2 |    0.75884  |     0.0500251 |       0.0500077 |    0.00131949 |    0.0377593 |         0.871552 |
| MLP 1-Regime (w256x256_d0.3)               | MLP_Global                |    0.72348  |  

## Hyperparameter Sweep Summary

27 MLP configs (width/depth, lr, dropout, weight-decay, batch size, activation, loss) trained in **both** families with 8 parallel H100 workers. Configs are ranked by **temporal holdout RMSE** (selection metric); test R² is reported for reference.

In [3]:
for family, fam_label in [("1regime", "1-regime (Global 54)"), ("2regime", "2-regime (Cluster c0=0,c1=10)")]:
    sub = df_sweep[df_sweep["family"] == family].sort_values("holdout_rmse").head(10)
    print(f"### Sweep Top-10 — {fam_label}")
    print(sub[["config_id", "hidden_sizes", "lr", "dropout", "weight_decay", "batch_size", "activation", "loss",
               "holdout_rmse", "test_r2", "test_rmse", "best_epoch", "train_time_s"]].to_markdown(index=False))
    print()

### Sweep Top-10 — 1-regime (Global 54)
| config_id               | hidden_sizes     |     lr |   dropout |   weight_decay |   batch_size | activation   | loss   |   holdout_rmse |   test_r2 |   test_rmse |   best_epoch |   train_time_s |
|:------------------------|:-----------------|-------:|----------:|---------------:|-------------:|:-------------|:-------|---------------:|----------:|------------:|-------------:|---------------:|
| w1024x1024              | [1024, 1024]     | 0.0003 |       0.1 |         0.0001 |          512 | silu         | mse    |      0.0679297 |  0.636094 |   0.0614512 |           40 |        9.06213 |
| w512x256x128_lr1e-3     | [512, 256, 128]  | 0.001  |       0.1 |         0.0001 |          512 | silu         | mse    |      0.0685209 |  0.695493 |   0.0562127 |            6 |        4.52451 |
| w512x256                | [512, 256]       | 0.0003 |       0.1 |         0.0001 |          512 | silu         | mse    |      0.0697557 |  0.655901 |   0.0597554

## Per-Regime Performance Breakdown

Test metrics by regime partition for the 2-regime MLP winners and the XGBoost references (from eval-1.1).

In [4]:
pcols = ["strategy_name", "model_name", "cluster", "n_train", "n_test", "r2", "rmse", "ubrmse", "bias", "mae"]
print("### Per-Regime Performance Breakdown")
print(df_per_regime[pcols].to_markdown(index=False))

### Per-Regime Performance Breakdown
| strategy_name             | model_name                                 |   cluster |   n_train |   n_test |       r2 |      rmse |    ubrmse |         bias |       mae |
|:--------------------------|:-------------------------------------------|----------:|----------:|---------:|---------:|----------:|----------:|-------------:|----------:|
| MLP_Global                | MLP 1-Regime (w1024x1024)                  |         0 |     13148 |     6620 | 0.636094 | 0.0614512 | 0.0517418 |  0.0331517   | 0.048217  |
| MLP_Global                | MLP 1-Regime (w512x256x128_lr1e-3)         |         0 |     13148 |     6620 | 0.695493 | 0.0562127 | 0.0482295 |  0.0288754   | 0.0443541 |
| MLP_Global                | MLP 1-Regime (w512x256)                    |         0 |     13148 |     6620 | 0.655901 | 0.0597554 | 0.0558301 |  0.0213004   | 0.0474596 |
| MLP_Clustering_V0_Full_k2 | MLP 2-Regime (w256x256_d0.3)               |         0 |      9562 |     

## Yearly Performance Breakdown

Test R² split by year (2023, 2024, 2025) for every leaderboard model.

In [5]:
ycols = ["model_name", "pooled_r2", "year_2023_r2", "year_2024_r2", "year_2025_r2"]
print("### Year-by-Year R² Breakdown")
print(df_summary[ycols].to_markdown(index=False))

### Year-by-Year R² Breakdown
| model_name                                 |   pooled_r2 |   year_2023_r2 |   year_2024_r2 |   year_2025_r2 |
|:-------------------------------------------|------------:|---------------:|---------------:|---------------:|
| Clustering_V0_Full_k2 (Winner c0=0, c1=10) |    0.81496  |       0.822971 |       0.783256 |       0.83029  |
| Global Single Model (54 Backbone)          |    0.77923  |       0.750748 |       0.770077 |       0.813582 |
| MLP 2-Regime (w256x256_d0.3)               |    0.75884  |       0.79551  |       0.777776 |       0.689432 |
| MLP 1-Regime (w256x256_d0.3)               |    0.72348  |       0.697243 |       0.763233 |       0.705439 |
| MLP 1-Regime (w512x256x128_lr1e-3)         |    0.695493 |       0.675986 |       0.750077 |       0.654163 |
| MLP 2-Regime (w256x256_lr1e-3)             |    0.689175 |       0.715474 |       0.753571 |       0.584897 |
| MLP 1-Regime (w512x256x128_d0.3)           |    0.683989 |       0.65748

## Extrapolation (OOD) Check

Test rows whose top-10 gain features fall outside the trainval [min, max] range are flagged as OOD. This directly probes the "XGBoost cannot extrapolate to unseen values" hypothesis.

In [6]:
print("### OOD Slice Metrics (best MLP per family vs XGBoost references)")
print(df_ood.to_markdown(index=False))

### OOD Slice Metrics (best MLP per family vs XGBoost references)
| model                       | slice           |    n |       r2 |      rmse |        bias |       mae |
|:----------------------------|:----------------|-----:|---------:|----------:|------------:|----------:|
| MLP 1regime (w1024x1024)    | all             | 6620 | 0.636094 | 0.0614512 |  0.0331517  | 0.048217  |
| MLP 1regime (w1024x1024)    | in_distribution | 6032 | 0.625317 | 0.0634403 |  0.0360583  | 0.0501638 |
| MLP 1regime (w1024x1024)    | ood             |  588 | 0.684408 | 0.0350402 |  0.00333499 | 0.0282463 |
| XGBoost Global (54)         | all             | 6620 | 0.77923  | 0.0478636 |  0.0105484  | 0.0370592 |
| XGBoost Global (54)         | in_distribution | 6032 | 0.780849 | 0.0485182 |  0.0145436  | 0.0373064 |
| XGBoost Global (54)         | ood             |  588 | 0.577509 | 0.0405427 | -0.0304369  | 0.0345237 |
| MLP 2regime (w256x256_d0.3) | all             | 6620 | 0.75884  | 0.0500251 |  0.001

## Timing

Per-config training time on the H100 (8 workers in parallel), plus total wall-clock.

In [7]:
print(f"### Total sweep wall time: {timing_log['sweep_wall_s']:.1f} s  |  eval wall time: {timing_log.get('eval_wall_s', float('nan')):.1f} s")
print(f"GPU: {timing_log['gpu']}")
print()
print("### Per-Config Timing (top-10 fastest/slowest by train time)")
sub = df_timing.sort_values("train_time_s")
print(sub.head(10)[["family", "config_id", "train_time_s", "wall_time_s", "epochs", "best_epoch", "holdout_rmse", "test_r2"]].to_markdown(index=False))
print()
print(sub.tail(10)[["family", "config_id", "train_time_s", "wall_time_s", "epochs", "best_epoch", "holdout_rmse", "test_r2"]].to_markdown(index=False))

### Total sweep wall time: 124.1 s  |  eval wall time: 4.7 s
GPU: {'device': 'NVIDIA H100 PCIe 80GB', 'n_parallel': 8}

### Per-Config Timing (top-10 fastest/slowest by train time)
| family   | config_id               |   train_time_s |   wall_time_s |   epochs |   best_epoch |   holdout_rmse |   test_r2 |
|:---------|:------------------------|---------------:|--------------:|---------:|-------------:|---------------:|----------:|
| 1regime  | w512x256x128_relu       |        4.39984 |       10.0103 |       31 |            6 |      0.0741264 |  0.623324 |
| 1regime  | w512x256x128_lr1e-3     |        4.52451 |       12.0118 |       31 |            6 |      0.0685209 |  0.695493 |
| 2regime  | w512x256_lr1e-3_d0      |        4.6298  |       10.006  |       57 |           32 |      0.0686792 |  0.48118  |
| 1regime  | w512x256x128            |        4.92985 |       12.001  |       30 |            5 |      0.0727581 |  0.5851   |
| 1regime  | w256x256_lr1e-3         |        5.11919 |  